# 第 3 课：策略梯度 REINFORCE 与自动微分

**学习目标**：理解为什么策略输出概率分布；用 JAX 实现随机策略的学习。

在连续控制任务中，策略一般不是“输出一个确定值”那么简单，而是输出分布的参数。离散动作中例如

$$\pi_\theta(a|s)=\operatorname{softmax}(f_\theta(s))_a.$$

优化目标为 $J(\theta)=\mathbb E_{\tau\sim\pi_\theta}[G(\tau)]$。关键的 score-function identity：

$$\nabla_\theta J(\theta)=\mathbb E_{\tau}\left[\sum_t\nabla_\theta\log\pi_\theta(a_t|s_t)\,G_t\right].$$

它不要求环境本身可微分，因此即使底层是不可微的 CFD 求解器也能训练。可微分代理模型则提供了**额外**的基于模型梯度优化可能性。

## 3.1 用一个单步控制问题先验证策略梯度

状态是观测到的扰动 $s\in\{-1,+1\}$，动作是 $a\in\{-1,+1\}$。当动作与扰动方向相反时奖励为 1；这是一个单步 MDP（contextual bandit），适合先排除长轨迹的干扰。

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
# theta=(w,b)；动作 +1 的概率 sigmoid(w*s+b)
def prob_plus(theta, states):
    return jax.nn.sigmoid(theta[0] * states + theta[1])

def neg_policy_gradient(theta, states, actions, rewards):
    p = prob_plus(theta, states)
    # Bernoulli log-probability: a = +1 or -1
    logp = jnp.where(actions == 1, jnp.log(p + 1e-8), jnp.log1p(-p + 1e-8))
    return -jnp.mean(logp * rewards)

loss_and_grad = jax.jit(jax.value_and_grad(neg_policy_gradient))
theta = jnp.array([0.0, 0.0])
curve = []
for step in range(250):
    s = rng.choice([-1, 1], size=128)
    p_plus = np.asarray(prob_plus(theta, jnp.asarray(s)))
    a = np.where(rng.random(128) < p_plus, 1, -1)
    reward = (a == -s).astype(float)
    loss, grad = loss_and_grad(theta, jnp.asarray(s), jnp.asarray(a), jnp.asarray(reward))
    theta = theta - 0.25 * grad
    curve.append(float(np.mean(reward)))
print("theta:", np.asarray(theta))
print("P(+1 | s=-1):", float(prob_plus(theta, jnp.array(-1.))))
print("P(+1 | s=+1):", float(prob_plus(theta, jnp.array(+1.))))
plt.figure(figsize=(6,3))
plt.plot(np.convolve(curve, np.ones(20)/20, mode='valid'))
plt.xlabel('Update'); plt.ylabel('Success rate (moving avg)'); plt.show()
assert float(prob_plus(theta, jnp.array(-1.))) > 0.75
assert float(prob_plus(theta, jnp.array(+1.))) < 0.25

## 3.2 为什么要引入 baseline？

用 $(G_t-b(s_t))$ 替代 $G_t$ 不改变期望策略梯度（在 baseline 不依赖所采样动作的条件下），但可以显著减小方差。进一步令

$$A^{\pi}(s,a)=Q^{\pi}(s,a)-V^{\pi}(s),$$

就形成了 Actor-Critic 的基础。它将“策略（Actor）”与“预估状态好坏（Critic）”分离。

注意这里的 REINFORCE 使用了采样动作的对数概率梯度，而**不是**对离散环境动作做反向传播。

### 练习

1. 从头推导：$\nabla_\theta p_\theta(\tau)=p_\theta(\tau)\nabla_\theta\log p_\theta(\tau)$。
2. 改成一个高斯策略 $a\sim\mathcal N(\mu_\theta(s),\sigma^2)$；写出其 `log_prob`。
3. 将单步环境改为多步，比较“只乘最终总回报”和“乘每个时间步的 $G_t$”的方差。